# Clasificador Cuántico Variacional (VQC) — Comparativo clásico vs cuántico

**Moons como default** (frontera no lineal) + selector `dataset="iris"` disponible.
Simulador numpy por defecto; Q# opcional vía `pip install .[quantum]` con validación cruzada.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from src.data.datasets import get_dataset
from src.data.preprocessing import normalize_to_pi, train_test_split_stratified
from src.classical.baseline import train_logistic_regression, evaluate
from src.training.loop import train, predict, predict_proba
from src.visualization.plots import plot_comparison, plot_loss_curve

DATASET = "moons"  # "moons" | "iris"
X_raw, y = get_dataset(DATASET, n_samples=200, noise=0.1, random_state=42)
X, scaler = normalize_to_pi(X_raw)
X_train, X_test, y_train, y_test = train_test_split_stratified(X, y, test_size=0.2, random_state=42)
print(f"{DATASET}: X={X.shape}, train={X_train.shape}, test={X_test.shape}")


In [ ]:
# Baseline clásico
clf = train_logistic_regression(X_train, y_train)
res = evaluate(clf, X_test, y_test)
print(f"LogReg accuracy: {res['accuracy']:.3f}")


In [ ]:
# VQC — Adam (default)
history = train(X_train, y_train, X_val=X_test, y_val=y_test, n_layers=3, lr=0.05, epochs=60, optimizer="adam", seed=42, verbose=True)
from src.training.loss import accuracy_from_probs
p_test = predict_proba(X_test, history["params_final"], n_layers=3)
acc_q = accuracy_from_probs(p_test, y_test)
print(f"VQC accuracy (adam): {acc_q:.3f}  loss final: {history['loss'][-1]:.4f}")


In [ ]:
# Comparativo SGD vs Adam (opcional, para README)
hist_sgd = train(X_train, y_train, n_layers=3, lr=0.05, epochs=60, optimizer="sgd", seed=42)
plt.figure()
plt.plot(history["loss"], label="adam")
plt.plot(hist_sgd["loss"], label="sgd")
plt.legend(); plt.xlabel("época"); plt.ylabel("BCE"); plt.title("Adam vs SGD — VQC"); plt.show()


In [ ]:
# Fronteras lado a lado
import matplotlib.pyplot as plt
from src.visualization.plots import plot_comparison
fig = plot_comparison(X, y, lambda Xg: clf.predict(Xg), lambda Xg: predict(Xg, history["params_final"], n_layers=3), save_path="../results/fronteras.png")
plt.show()


In [ ]:
# Curva de pérdida
fig = plot_loss_curve(history, save_path="../results/loss_curve.png")
plt.show()
print("Si ves estancamiento plano, es barren plateau — no bug.")


## Conclusión honesta
Para este tamaño de dataset lo clásico probablemente gana en exactitud y velocidad. La pregunta no es "¿ganó lo cuántico?" sino "¿bajo qué condiciones podría y por qué no hoy?" — ver README.
